In [0]:

%pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg openai pydicom
dbutils.library.restartPython()

In [0]:
# Cell 1: Config & Imports

dbutils.widgets.text("input_folder", "Hackathon3/", "Input DICOM Folder")
dbutils.widgets.text("deployment_name", "gpt-4.1-mini-b", "Model Deployment Name")
dbutils.widgets.text("poll_interval_seconds", "60", "Batch Poll Interval (seconds)")
dbutils.widgets.text("max_batch_size_mb", "150", "Max JSONL file size per batch (MB)")

from openai import OpenAI
import json
import time
import datetime
import os
import glob
import gc

api_key = dbutils.secrets.get(scope="adc_store", key="hackathon_gpt41mini_api_key")
endpoint = "https://telefonica-hackathon-2026.cognitiveservices.azure.com/openai/v1/"
deployment_name = dbutils.widgets.get("deployment_name")
poll_interval = int(dbutils.widgets.get("poll_interval_seconds"))
max_batch_size_bytes = int(dbutils.widgets.get("max_batch_size_mb")) * 1024 * 1024

client = OpenAI(base_url=endpoint, api_key=api_key)

print(f"Endpoint: {endpoint}")
print(f"Deployment: {deployment_name}")
print(f"Poll interval: {poll_interval}s")
print(f"Max batch size: {max_batch_size_bytes / (1024*1024):.0f} MB")

In [0]:
# Cell 2: List DICOM Files

input_dir = f"/Volumes/1_inland/sectra/vone/{dbutils.widgets.get('input_folder')}"

dicom_paths = []
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith(".dcm") and "DICOMDIR" not in f:
            dicom_paths.append(os.path.join(root, f))

print(f"Found {len(dicom_paths)} DICOM files")

In [0]:
# Cell 3: Generate Batch JSONL
#
# Processes all DICOMs sequentially on the driver with memory-safe frame extraction.
# For multi-frame DICOMs: uses get_frame() to extract only the first frame's
# compressed bytes (~107KB) instead of decompressing all frames (can be 748MB+).
# Encoding time is ~3-5 min for 20k files - negligible vs batch API turnaround.

import pydicom
from pydicom.encaps import get_frame, generate_pixel_data_frame
import numpy as np
from PIL import Image
from io import BytesIO
import base64

SYSTEM_PROMPT = (
    "You are a medical image PII detector. Analyze the image and determine: "
    "(1) whether any text is visible, "
    "(2) whether that text contains personal information "
    "(e.g. patient name, date of birth, scan date, ID numbers, address). "
    "Return confidence_score as a float from 0.0 to 1.0 where 0.0 means certainly no PII "
    "and 1.0 means certainly PII present. "
    "If no text is detected, set confidence_score to 0.0 and detected_text_summary to an empty string."
)

USER_PROMPT = "Analyze this medical image for visible text and personal information."

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "PIIDetectionResult",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "is_text_detected": {"type": "boolean"},
                "is_personal_information_detected": {"type": "boolean"},
                "confidence_score": {"type": "number"},
                "detected_text_summary": {"type": "string"},
            },
            "required": [
                "is_text_detected",
                "is_personal_information_detected",
                "confidence_score",
                "detected_text_summary",
            ],
            "additionalProperties": False,
        },
    },
}

folder_name = str(dbutils.widgets.get("input_folder")).replace("/", "")
staging_dir = f"/Volumes/1_inland/sectra/vone/{folder_name}_batch_staging"
os.makedirs(staging_dir, exist_ok=True)
for old_file in glob.glob(f"{staging_dir}/*.jsonl"):
    os.remove(old_file)

id_to_path = {}
dicom_read_failures = []

# Batch file management
batch_jsonl_files = []
batch_file_idx = 0
current_batch_path = f"{staging_dir}/batch-{batch_file_idx:03d}.jsonl"
current_batch_handle = open(current_batch_path, "w")
current_batch_size = 0
current_batch_count = 0


def rotate_batch():
    global batch_file_idx, current_batch_path, current_batch_handle
    global current_batch_size, current_batch_count
    current_batch_handle.close()
    if current_batch_count > 0:
        batch_jsonl_files.append((current_batch_path, current_batch_count))
    else:
        os.remove(current_batch_path)
    batch_file_idx += 1
    current_batch_path = f"{staging_dir}/batch-{batch_file_idx:03d}.jsonl"
    current_batch_handle = open(current_batch_path, "w")
    current_batch_size = 0
    current_batch_count = 0


def extract_first_frame(file_path):
    """Extract frame 0 from any DICOM. Memory-safe for multi-frame files."""
    # Read header to check frame count
    ds_header = pydicom.dcmread(file_path, stop_before_pixels=True)
    rows = ds_header.Rows
    cols = ds_header.Columns
    bits = ds_header.BitsAllocated
    num_frames = int(ds_header.NumberOfFrames) if "NumberOfFrames" in ds_header else 1
    samples = ds_header.SamplesPerPixel if "SamplesPerPixel" in ds_header else 1
    del ds_header

    # Estimate full uncompressed size
    full_size_mb = rows * cols * (bits // 8) * num_frames * samples / (1024**2)

    ds = pydicom.dcmread(file_path, stop_before_pixels=False)

    if num_frames > 1 and full_size_mb > 50:
        # Large multi-frame: extract only first compressed frame
        is_encapsulated = (
            hasattr(ds["PixelData"], "is_undefined_length")
            and ds["PixelData"].is_undefined_length
        )
        if is_encapsulated:
            try:
                frame_bytes = get_frame(ds.PixelData, 0)
            except Exception:
                frame_bytes = next(generate_pixel_data_frame(ds.PixelData, num_frames))
            del ds
            gc.collect()
            return np.array(Image.open(BytesIO(frame_bytes)))
        else:
            # Uncompressed multi-frame: slice raw bytes for frame 0
            frame_nbytes = rows * cols * (bits // 8) * samples
            raw = ds.PixelData[:frame_nbytes]
            del ds
            dtype = np.uint8 if bits == 8 else np.uint16
            return np.frombuffer(raw, dtype=dtype).reshape(rows, cols)
    else:
        # Single frame or small multi-frame: safe to decompress all
        arr = ds.pixel_array
        if arr.ndim == 3 and num_frames > 1:
            frame = arr[0].copy()
            del arr
        else:
            frame = arr
        del ds
        return frame


def frame_to_b64(frame_arr):
    """Resize frame to 512x512 and encode as base64 PNG."""
    img = Image.fromarray(frame_arr)
    del frame_arr
    img = img.resize((512, 512), Image.LANCZOS)

    arr_small = np.array(img, dtype=np.float32)
    del img
    arr_min, arr_max = arr_small.min(), arr_small.max()
    if arr_max > arr_min:
        arr_small = ((arr_small - arr_min) * (255.0 / (arr_max - arr_min))).astype(np.uint8)
    else:
        arr_small = np.zeros(arr_small.shape, dtype=np.uint8)

    img_out = Image.fromarray(arr_small)
    del arr_small
    buf = BytesIO()
    img_out.save(buf, format="PNG", optimize=True)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    buf.close()
    del img_out
    return b64


start_time = time.time()

for idx, path in enumerate(dicom_paths):
    custom_id = f"task-{idx}"
    id_to_path[custom_id] = path

    try:
        frame = extract_first_frame(path)
        b64 = frame_to_b64(frame)
        del frame

        line = json.dumps({
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": deployment_name,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": USER_PROMPT},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{b64}",
                                    "detail": "low",
                                },
                            },
                        ],
                    },
                ],
                "max_tokens": 200,
                "response_format": RESPONSE_FORMAT,
            },
        }) + "\n"

        line_bytes = len(line.encode("utf-8"))
        del b64

        if current_batch_size > 0 and current_batch_size + line_bytes > max_batch_size_bytes:
            rotate_batch()

        current_batch_handle.write(line)
        current_batch_size += line_bytes
        current_batch_count += 1
        del line

    except Exception as e:
        dicom_read_failures.append({"path": path, "error": str(e)[:1000]})

    gc.collect()

    if (idx + 1) % 500 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        eta = (len(dicom_paths) - idx - 1) / rate
        print(f"  {idx + 1}/{len(dicom_paths)} ({rate:.1f} files/s, ETA {eta/60:.1f} min)")

# Close final batch file
current_batch_handle.close()
if current_batch_count > 0:
    batch_jsonl_files.append((current_batch_path, current_batch_count))
elif os.path.exists(current_batch_path):
    os.remove(current_batch_path)

elapsed = time.time() - start_time
print(f"\nDICOM processing complete in {elapsed/60:.1f} minutes:")
print(f"  Successfully encoded: {len(id_to_path) - len(dicom_read_failures)}")
print(f"  Read failures: {len(dicom_read_failures)}")
print(f"\nGenerated {len(batch_jsonl_files)} batch file(s):")
for i, (bf, count) in enumerate(batch_jsonl_files):
    sz = os.path.getsize(bf) / (1024 * 1024)
    print(f"  Batch {i+1}: {count} requests, {sz:.1f} MB")

# Free heavy imports
del pydicom, np, Image, BytesIO, base64
gc.collect()

In [0]:
# Cell 4: Upload & submit batch jobs

batch_jobs = []
for i, (batch_path, count) in enumerate(batch_jsonl_files):
    file_response = client.files.create(
        file=open(batch_path, "rb"),
        purpose="batch",
        extra_body={"expires_after": {"seconds": 1209600, "anchor": "created_at"}},
    )
    file_id = file_response.id
    print(f"Batch {i+1}/{len(batch_jsonl_files)}: Uploaded file {file_id} ({count} requests)")

    batch_response = client.batches.create(
        input_file_id=file_id,
        endpoint="/chat/completions",
        completion_window="24h",
    )
    batch_jobs.append((batch_response.id, batch_response))
    print(f"  Job created: {batch_response.id} (status: {batch_response.status})")

print(f"\nSubmitted {len(batch_jobs)} batch job(s)")

In [0]:
# Cell 5: Poll all batch jobs for completion

completed_batches = {}
pending = {bid: resp for bid, resp in batch_jobs}

while pending:
    time.sleep(poll_interval)
    still_pending = {}
    for batch_id, _ in pending.items():
        batch_response = client.batches.retrieve(batch_id)
        status = batch_response.status
        counts = batch_response.request_counts
        print(
            f"{datetime.datetime.now():%H:%M:%S} | "
            f"Batch {batch_id[:16]}... | "
            f"Status: {status} | "
            f"Done: {counts.completed}/{counts.total} | "
            f"Failed: {counts.failed}"
        )
        if status in ("completed", "failed", "cancelled"):
            completed_batches[batch_id] = batch_response
            if status == "failed":
                print(f"  FAILED:")
                for error in batch_response.errors.data:
                    print(f"    [{error.code}] {error.message}")
            elif status == "cancelled":
                print(f"  CANCELLED")
        else:
            still_pending[batch_id] = batch_response
    pending = still_pending
    if pending:
        print(f"  --- {len(pending)} job(s) still running ---")

print(f"\nAll {len(completed_batches)} batch job(s) finished.")

In [0]:
# Cell 6: Download & parse results from all batches

results = []


def parse_output_line(line):
    resp = json.loads(line)
    custom_id = resp["custom_id"]
    path = id_to_path.get(custom_id, custom_id)

    if resp.get("error"):
        return {
            "input_path": path,
            "is_loaded_ok": True,
            "is_text_detected": None,
            "is_pii_detected": None,
            "confidence_score": None,
            "detected_text_summary": None,
            "llm_raw_result": None,
            "error_message": json.dumps(resp["error"]),
        }

    try:
        raw_content = resp["response"]["body"]["choices"][0]["message"]["content"]
        parsed = json.loads(raw_content)
        return {
            "input_path": path,
            "is_loaded_ok": True,
            "is_text_detected": parsed["is_text_detected"],
            "is_pii_detected": parsed["is_personal_information_detected"],
            "confidence_score": float(parsed["confidence_score"]),
            "detected_text_summary": parsed["detected_text_summary"],
            "llm_raw_result": raw_content,
            "error_message": None,
        }
    except Exception as e:
        return {
            "input_path": path,
            "is_loaded_ok": True,
            "is_text_detected": None,
            "is_pii_detected": None,
            "confidence_score": None,
            "detected_text_summary": None,
            "llm_raw_result": str(resp.get("response", "")),
            "error_message": f"Parse error: {str(e)[:500]}",
        }


seen_paths = set()

for batch_id, batch_response in completed_batches.items():
    output_file_id = batch_response.output_file_id
    if output_file_id:
        output_content = client.files.content(output_file_id).text
        for line in output_content.strip().split("\n"):
            if not line:
                continue
            result = parse_output_line(line)
            results.append(result)
            seen_paths.add(result["input_path"])

    error_file_id = batch_response.error_file_id
    if error_file_id:
        error_content = client.files.content(error_file_id).text
        for line in error_content.strip().split("\n"):
            if not line:
                continue
            resp = json.loads(line)
            custom_id = resp["custom_id"]
            path = id_to_path.get(custom_id, custom_id)
            if path not in seen_paths:
                results.append({
                    "input_path": path,
                    "is_loaded_ok": True,
                    "is_text_detected": None,
                    "is_pii_detected": None,
                    "confidence_score": None,
                    "detected_text_summary": None,
                    "llm_raw_result": None,
                    "error_message": json.dumps(resp.get("error", resp)),
                })
                seen_paths.add(path)

for fail in dicom_read_failures:
    results.append({
        "input_path": fail["path"],
        "is_loaded_ok": False,
        "is_text_detected": None,
        "is_pii_detected": None,
        "confidence_score": None,
        "detected_text_summary": None,
        "llm_raw_result": None,
        "error_message": fail["error"],
    })

print(f"Total results: {len(results)}")
print(f"  Successful: {sum(1 for r in results if r['error_message'] is None)}")
print(f"  Errors: {sum(1 for r in results if r['error_message'] is not None)}")

In [0]:
# Cell 7: Build DataFrame & Save

from pyspark.sql.types import StructType, StructField, StringType, BooleanType, FloatType

output_schema = StructType([
    StructField("input_path", StringType(), True),
    StructField("is_loaded_ok", BooleanType(), True),
    StructField("is_text_detected", BooleanType(), True),
    StructField("is_pii_detected", BooleanType(), True),
    StructField("confidence_score", FloatType(), True),
    StructField("detected_text_summary", StringType(), True),
    StructField("llm_raw_result", StringType(), True),
    StructField("error_message", StringType(), True),
])

results_df = spark.createDataFrame(results, schema=output_schema)

output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('input_folder')).replace('/', '')}_batch_output.parquet"
results_df.write.mode("overwrite").parquet(output_path)
print(f"Results saved to: {output_path}")

import shutil
shutil.rmtree(staging_dir, ignore_errors=True)
print(f"Cleaned up staging dir: {staging_dir}")

display(results_df)

In [0]:
output_path = "/Volumes/1_inland/sectra/vone/Hackathon3_batch_output.parquet"
df_out2 = spark.read.parquet(output_path).filter(" llm_raw_result IS NOT NULL")
display(df_out2.limit(1000))
print(df_out2.count())